
# 🥉 NB01 — Bronze Layer: Data Ingestion
## Investor Intelligence Platform — FIIs Brasil 🇧🇷
### CRISP-DM Phase: Data Understanding

| | |
|---|---|
| **Fontes** | 20 portais editoriais + Reddit (fonte #21) |
| **Output** | `data/external/*.parquet` — Bronze **frozen** |
| **Schema** | Bronze Contract — 17 campos |
| **Estratégia** | RSS-first → Scraping fallback → Reddit 3-tier |

> 🔒 **Regra do freeze:** Esta célula final grava `data/external/`.
> NB02–NB07 lêem exclusivamente `data/external/`. Nenhum notebook downstream faz requests HTTP.

## 📋 Fontes — 21 no Total

| Grupo | Contagem | Método |
|-------|----------|--------|
| RSS primário (settings.py) | 6 | feedparser |
| RSS suplementar (NB01) | 4 | feedparser |
| Portal scraping | 10 | requests + BeautifulSoup |
| Reddit | 1 | PRAW / API pública / parquet frozen |
| **Total** | **21** | |

## 📦 Seção 1 — Imports

In [2]:

import sys
import os
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import re, json, time, random, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import pandas as pd
import numpy as np
import feedparser
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')

CURRENT_DIR  = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'✅ Imports loaded')
print(f'   PROJECT_ROOT: {PROJECT_ROOT}')

✅ Imports loaded
   PROJECT_ROOT: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform


## ⚙️ Seção 2 — Configuração

In [3]:

from config.settings import (
    BRONZE_DIR, EXTERNAL_DIR, RANDOM_SEED,
    REQUEST_TIMEOUT, MAX_RETRIES, RETRY_BACKOFF, RATE_LIMIT_DELAY,
    USER_AGENTS, RSS_FEEDS, FII_FILTER_TERMS,
    REDDIT_CLIENT_ID, REDDIT_CLIENT_SECRET, REDDIT_USER_AGENT,
    REDDIT_SUBREDDITS, REDDIT_API_AVAILABLE,
    SPARK_DRIVER_MEMORY, SPARK_APP_NAME, SPARK_SHUFFLE_PARTS,
)

print(f'✅ config.settings carregado')
print(f'   EXTERNAL_DIR:         {EXTERNAL_DIR}')
print(f'   RSS_FEEDS:            {len(RSS_FEEDS)}')
print(f'   FII_FILTER_TERMS:     {len(FII_FILTER_TERMS)}')
print(f'   REDDIT_API_AVAILABLE: {REDDIT_API_AVAILABLE}')
print(f'   REDDIT_SUBREDDITS:    {REDDIT_SUBREDDITS}')
print(f'   RANDOM_SEED:          {RANDOM_SEED}')

✅ config.settings carregado
   EXTERNAL_DIR:         /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/external
   RSS_FEEDS:            6
   FII_FILTER_TERMS:     27
   REDDIT_API_AVAILABLE: False
   REDDIT_SUBREDDITS:    ['investimentos', 'farialimabets']
   RANDOM_SEED:          42


## 📝 Seção 3 — Logger e Fontes Suplementares

In [4]:

from src.utils.logger import (
    ingestion_logger, log_source_success, log_source_failure,
    log_retry, log_timeout, log_duplicate, log_dataset_frozen,
    log_quality_check, log_spark_start,
)

logger = ingestion_logger()
print('✅ src.utils.logger carregado')

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
print(f'✅ RANDOM_SEED = {RANDOM_SEED} aplicado')

# ── RSS suplementares (4 feeds — definidos em NB01, não em settings.py) ───────
SUPPLEMENTAL_RSS_FEEDS: List[str] = [
    'https://www.sunoresearch.com.br/feed/',           # 7. Suno Research
    'https://einvestidor.estadao.com.br/feed',         # 8. E-Investidor Estadão
    'https://neofeed.com.br/feed/',                    # 9. NeoFeed
    'https://blog.toroinvestimentos.com.br/feed/',     # 10. Toro (RSS-first; scraping fallback)
]

# ── Portal scraping targets (10 portais — sem RSS) ────────────────────────────
PORTAL_TARGETS: List[Dict[str, str]] = [
    {'name': 'fundsexplorer.com.br',    'url': 'https://www.fundsexplorer.com.br/ranking'},
    {'name': 'statusinvest.com.br',     'url': 'https://statusinvest.com.br/fundos-imobiliarios'},
    {'name': 'clubefii.com.br',         'url': 'https://www.clubefii.com.br/'},
    {'name': 'fiis.com.br',             'url': 'https://fiis.com.br/'},
    {'name': 'portaldofii.com.br',      'url': 'https://portaldofii.com.br/'},
    {'name': 'investidor10.com.br',     'url': 'https://investidor10.com.br/fiis/'},
    {'name': 'euqueroinvestir.com',     'url': 'https://euqueroinvestir.com/fundos-imobiliarios/'},
    {'name': 'borainvestir.b3.com.br',  'url': 'https://borainvestir.b3.com.br/'},
    {'name': 'conteudos.xpi.com.br',    'url': 'https://conteudos.xpi.com.br/'},
    {'name': 'br.investing.com',        'url': 'https://br.investing.com/news/stock-market-news'},
]

_total = len(RSS_FEEDS) + len(SUPPLEMENTAL_RSS_FEEDS) + len(PORTAL_TARGETS)
print(f'\n✅ RSS primário (settings.py): {len(RSS_FEEDS)} feeds')
print(f'✅ RSS suplementar (NB01):     {len(SUPPLEMENTAL_RSS_FEEDS)} feeds')
print(f'✅ Portal targets:             {len(PORTAL_TARGETS)} portais')
print(f'✅ Editorial total:            {_total}/20')
assert _total == 20, f'Esperado 20 editoriais, obteve {_total}'
print(f'✅ + Reddit (Fonte #21) = 21 fontes monitoradas ✅')

# Garantir diretório de saída
Path(EXTERNAL_DIR).mkdir(parents=True, exist_ok=True)

✅ src.utils.logger carregado
✅ RANDOM_SEED = 42 aplicado

✅ RSS primário (settings.py): 6 feeds
✅ RSS suplementar (NB01):     4 feeds
✅ Portal targets:             10 portais
✅ Editorial total:            20/20
✅ + Reddit (Fonte #21) = 21 fontes monitoradas ✅



## 🗺️ Seção 4 — Bronze Schema Contract (17 Campos)

| Campo-chave | Derivação | Notas |
|-------------|-----------|-------|
| `article_id` | `SHA-256(url)` | Chave primária determinística |
| `content_hash` | `MD5(title+content[:500])` | Dedup de conteúdo quase-duplicado |
| `published_at` | RSS: data do feed; Scraping: **`None`**; Reddit: UTC ISO da criação | Nunca preenchido com `collected_at` |

In [5]:

BRONZE_COLUMNS: List[str] = [
    'article_id', 'source', 'source_type', 'title', 'content',
    'summary', 'url', 'author', 'published_at', 'collected_at',
    'language', 'tags', 'query_used', 'ingestion_method',
    'raw_html', 'content_hash', 'metadata_json',
]

_BRONZE_DEFAULTS: Dict = {
    'article_id': '',     'source': 'unknown',  'source_type': 'unknown',
    'title': '[NO TITLE]','content': None,       'summary': None,
    'url': '',            'author': None,        'published_at': None,
    'collected_at': '',   'language': 'pt-br',   'tags': None,
    'query_used': None,   'ingestion_method': 'unknown',
    'raw_html': None,     'content_hash': '',    'metadata_json': None,
}


def make_article_id(url: str) -> str:
    return hashlib.sha256(url.strip().encode('utf-8')).hexdigest()

def make_content_hash(title: str, content: str) -> str:
    combined = f"{title or ''}{(content or '')[:500]}"
    return hashlib.md5(combined.encode('utf-8')).hexdigest()

def normalize_url(url: str) -> str:
    url = url.strip().rstrip('/')
    url = re.sub(r'[?&](utm_source|utm_medium|utm_campaign|utm_content|ref|fbclid|gclid)[^&]*', '', url)
    return re.sub(r'[?&]$', '', url)

def is_fii_relevant(title: str, body: str) -> bool:
    text = f'{title} {body}'.lower()
    return any(term in text for term in FII_FILTER_TERMS)

def matched_filter_term(title: str, body: str) -> Optional[str]:
    text = f'{title} {body}'.lower()
    return next((t for t in FII_FILTER_TERMS if t in text), None)

def strip_html(raw: str) -> str:
    if not raw: return ''
    soup = BeautifulSoup(raw, 'html.parser')
    for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside', 'form']):
        tag.decompose()
    return re.sub(r'\s+', ' ', soup.get_text(separator=' ')).strip()

def enforce_schema(record: Dict) -> Dict:
    merged = {**_BRONZE_DEFAULTS, **record}
    if not merged.get('collected_at'):
        merged['collected_at'] = datetime.now(timezone.utc).isoformat()
    return {col: merged[col] for col in BRONZE_COLUMNS}

print(f'✅ Bronze schema: {len(BRONZE_COLUMNS)} campos')

✅ Bronze schema: 17 campos


## 🔄 Seção 5 — HTTP Client com Retry Exponencial

In [6]:

def get_headers() -> Dict[str, str]:
    return {
        'User-Agent':      random.choice(USER_AGENTS),
        'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'pt-BR,pt;q=0.9,en;q=0.7',
        'Accept-Encoding': 'gzip, deflate',
        'Connection':      'keep-alive',
        'DNT':             '1',
    }

def fetch_with_retry(url: str, source: str) -> Optional[requests.Response]:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            return resp
        except requests.exceptions.Timeout:
            log_timeout(logger, source, REQUEST_TIMEOUT)
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF ** attempt)
        except requests.exceptions.HTTPError as exc:
            log_source_failure(logger, source, f'HTTP {exc.response.status_code}')
            return None
        except requests.exceptions.RequestException as exc:
            log_source_failure(logger, source, str(exc))
            if attempt < MAX_RETRIES:
                log_retry(logger, source, attempt, MAX_RETRIES)
                time.sleep(RETRY_BACKOFF ** attempt)
    return None

print('✅ HTTP client pronto')
print(f'   timeout={REQUEST_TIMEOUT}s | retries={MAX_RETRIES} | backoff={RETRY_BACKOFF}x | delay={RATE_LIMIT_DELAY}s')

✅ HTTP client pronto
   timeout=20s | retries=3 | backoff=2x | delay=1.0s


## 📡 Seção 6 — Ingestão RSS (Priority 1)

In [7]:

def ingest_rss_feed(feed_url: str) -> List[Dict]:
    records: List[Dict] = []
    source = feed_url.split('//')[1].split('/')[0].replace('www.', '')

    try:
        feed = feedparser.parse(feed_url)
        if feed.bozo and not feed.entries:
            log_source_failure(logger, source, f'RSS bozo: {feed.bozo_exception}')
            return records

        for entry in feed.entries:
            url_raw = entry.get('link', '').strip()
            title   = entry.get('title', '').strip()
            if not url_raw or not title:
                continue

            summary_raw    = entry.get('summary', entry.get('description', ''))
            summary        = strip_html(summary_raw)
            content_blocks = entry.get('content', [])
            content_raw    = content_blocks[0].get('value', '') if content_blocks else ''
            content        = strip_html(content_raw) if content_raw else summary

            if not is_fii_relevant(title, content):
                continue

            url  = normalize_url(url_raw)
            tags = ', '.join(
                t.get('term', '') for t in entry.get('tags', []) if t.get('term')
            ) or None

            # published_at: from feed entry (real date) — RSS provides actual dates
            pub_at = entry.get('published') or entry.get('updated') or None

            records.append(enforce_schema({
                'article_id':       make_article_id(url),
                'source':           source,
                'source_type':      'rss',
                'title':            title,
                'content':          content or None,
                'summary':          summary[:500] if summary else None,
                'url':              url,
                'author':           entry.get('author') or None,
                'published_at':     pub_at,
                'collected_at':     datetime.now(timezone.utc).isoformat(),
                'language':         'pt-br',
                'tags':             tags,
                'query_used':       matched_filter_term(title, content),
                'ingestion_method': 'feedparser',
                'raw_html':         None,
                'content_hash':     make_content_hash(title, content),
                'metadata_json':    json.dumps({'feed_url': feed_url, 'feed_bozo': bool(feed.bozo)}),
            }))

    except Exception as exc:
        log_source_failure(logger, source, str(exc))

    return records

print('✅ ingest_rss_feed() definida')

✅ ingest_rss_feed() definida


## 🌐 Seção 7 — Portal Scraping (Priority 2)

In [8]:

def scrape_portal(target: Dict[str, str]) -> List[Dict]:
    records: List[Dict] = []
    source  = target['name']
    resp    = fetch_with_retry(target['url'], source)
    if resp is None:
        return records

    try:
        soup = BeautifulSoup(resp.content, 'html.parser')
        seen_hashes: set = set()

        for elem in soup.find_all(['article', 'div', 'section', 'h2', 'h3', 'p'], limit=150):
            text = elem.get_text(separator=' ', strip=True)
            if len(text) < 80:
                continue
            if not is_fii_relevant(text, ''):
                continue

            title_text = text[:120].strip()
            c_hash     = make_content_hash(title_text, text)
            if c_hash in seen_hashes:
                log_duplicate(logger, title_text)
                continue
            seen_hashes.add(c_hash)

            link_tag = elem.find('a', href=True)
            href     = link_tag['href'].strip() if link_tag else ''
            if href.startswith('/'):
                href = f'https://{source}{href}'
            elif not href.startswith('http'):
                href = f"{target['url'].rstrip('/')}/{href.lstrip('/')}" if href else f"{target['url']}#s{len(records)}"
            url = normalize_url(href)

            # published_at = None for scraped content — no reliable publication date
            records.append(enforce_schema({
                'article_id':       make_article_id(url),
                'source':           source,
                'source_type':      'scraping',
                'title':            title_text,
                'content':          text,
                'summary':          text[:300],
                'url':              url,
                'author':           None,
                'published_at':     None,   # scraped content has no real pub date
                'collected_at':     datetime.now(timezone.utc).isoformat(),
                'language':         'pt-br',
                'tags':             None,
                'query_used':       matched_filter_term(title_text, text),
                'ingestion_method': 'requests+bs4',
                'raw_html':         str(elem)[:2000],
                'content_hash':     c_hash,
                'metadata_json':    json.dumps({
                    'status_code': resp.status_code,
                    'elapsed_ms':  int(resp.elapsed.total_seconds() * 1000),
                    'encoding':    resp.encoding,
                }),
            }))

    except Exception as exc:
        log_source_failure(logger, source, str(exc))

    return records

print('✅ scrape_portal() definida — published_at=None para conteúdo raspado')

✅ scrape_portal() definida — published_at=None para conteúdo raspado


## 🔁 Seção 8 — Execução RSS + Fallbacks

In [9]:

_all_rss = RSS_FEEDS + SUPPLEMENTAL_RSS_FEEDS
print(f'\n📡 RSS INGESTION — Priority 1')
print(f'   {len(RSS_FEEDS)} primários + {len(SUPPLEMENTAL_RSS_FEEDS)} suplementares = {len(_all_rss)} feeds\n')

rss_records: List[Dict] = []
rss_stats:   Dict[str, int] = {}

for feed_url in _all_rss:
    source   = feed_url.split('//')[1].split('/')[0].replace('www.', '')
    articles = ingest_rss_feed(feed_url)
    rss_records.extend(articles)
    rss_stats[source] = len(articles)
    icon = '✅' if articles else '⚠️ '
    print(f'   {icon} {source:<38} {len(articles):>4} artigos')
    time.sleep(RATE_LIMIT_DELAY)

df_rss     = (pd.DataFrame(rss_records, columns=BRONZE_COLUMNS)
              if rss_records else pd.DataFrame(columns=BRONZE_COLUMNS))
n_rss_raw  = len(df_rss)
df_rss     = df_rss.drop_duplicates('article_id').drop_duplicates('content_hash')

# ── Toro fallback ──────────────────────────────────────────────────────────────
if rss_stats.get('blog.toroinvestimentos.com.br', 0) == 0:
    print('   ⚠️  Toro RSS = 0 → ativando scraping fallback')
    _fb = scrape_portal({'name': 'blog.toroinvestimentos.com.br',
                         'url': 'https://blog.toroinvestimentos.com.br/'})
    if _fb:
        df_rss = pd.concat([df_rss, pd.DataFrame(_fb, columns=BRONZE_COLUMNS)], ignore_index=True)
        df_rss = df_rss.drop_duplicates('article_id').drop_duplicates('content_hash')
        print(f'   ✅ Toro scraping fallback: {len(_fb)} snippets')

# ── Empiricus fallback ─────────────────────────────────────────────────────────
if rss_stats.get('empiricus.com.br', 0) == 0:
    print('   ⚠️  Empiricus RSS = 0 → ativando scraping fallback')
    _fb = scrape_portal({'name': 'empiricus.com.br', 'url': 'https://empiricus.com.br/'})
    if _fb:
        df_rss = pd.concat([df_rss, pd.DataFrame(_fb, columns=BRONZE_COLUMNS)], ignore_index=True)
        df_rss = df_rss.drop_duplicates('article_id').drop_duplicates('content_hash')
        print(f'   ✅ Empiricus scraping fallback: {len(_fb)} snippets')

print(f'\n   📊 RSS total: {len(df_rss):,}  (removidos {n_rss_raw - len(df_rss)} duplicatas)')


📡 RSS INGESTION — Priority 1
   6 primários + 4 suplementares = 10 feeds

   ✅ infomoney.com.br                          1 artigos
   ✅ empiricus.com.br                         32 artigos
   ✅ moneytimes.com.br                         4 artigos
   ✅ seudinheiro.com                           6 artigos
   ✅ exame.com                                 1 artigos
   ✅ cnnbrasil.com.br                         17 artigos
   ✅ sunoresearch.com.br                       6 artigos
   ✅ einvestidor.estadao.com.br                2 artigos
   ⚠️  neofeed.com.br                            0 artigos
   ⚠️  blog.toroinvestimentos.com.br             0 artigos
   ⚠️  Toro RSS = 0 → ativando scraping fallback

   📊 RSS total: 69  (removidos 0 duplicatas)


## 🌐 Seção 9 — Execução Portal Scraping + Fallback Portal do FII

In [10]:

print('\n🌐 PORTAL SCRAPING — Priority 2')
print(f'   {len(PORTAL_TARGETS)} portais | delay={RATE_LIMIT_DELAY * 1.5:.1f}s\n')

portal_records: List[Dict] = []
portal_stats:   Dict[str, int] = {}

for target in PORTAL_TARGETS:
    source   = target['name']
    articles = scrape_portal(target)
    portal_records.extend(articles)
    portal_stats[source] = len(articles)
    icon = '✅' if articles else '⚠️ '
    print(f'   {icon} {source:<38} {len(articles):>4} snippets')
    time.sleep(RATE_LIMIT_DELAY * 1.5)

df_portals    = (pd.DataFrame(portal_records, columns=BRONZE_COLUMNS)
                 if portal_records else pd.DataFrame(columns=BRONZE_COLUMNS))
n_portal_raw  = len(df_portals)
if not df_portals.empty:
    df_portals = df_portals.drop_duplicates('article_id').drop_duplicates('content_hash')

# ── Portal do FII RSS fallback ─────────────────────────────────────────────────
if portal_stats.get('portaldofii.com.br', 0) == 0:
    print('   ⚠️  portaldofii.com.br scraping falhou → tentando RSS fallback')
    _pfii = ingest_rss_feed('https://portaldofii.com.br/feed/')
    if _pfii:
        _df_pfii   = pd.DataFrame(_pfii, columns=BRONZE_COLUMNS)
        df_portals = pd.concat([df_portals, _df_pfii], ignore_index=True)
        df_portals = df_portals.drop_duplicates('article_id').drop_duplicates('content_hash')
        print(f'   ✅ Portal do FII RSS fallback: {len(_pfii)} artigos')

print(f'\n   📊 Portals total: {len(df_portals):,}  (removidos {n_portal_raw - len(df_portals)} duplicatas)')


🌐 PORTAL SCRAPING — Priority 2
   10 portais | delay=1.5s

   ✅ fundsexplorer.com.br                     18 snippets
   ✅ statusinvest.com.br                      25 snippets
   ✅ clubefii.com.br                          10 snippets
   ✅ fiis.com.br                              15 snippets
2026-06-14T15:38:10 | WARNING  | fii_pipeline.ingestion | SOURCE_FAIL | portaldofii.com.br | HTTPSConnectionPool(host='portaldofii.com.br', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='portaldofii.com.br', port=443): Failed to resolve 'portaldofii.com.br' ([Errno 8] nodename nor servname provided, or not known)"))
2026-06-14T15:38:12 | WARNING  | fii_pipeline.ingestion | SOURCE_FAIL | portaldofii.com.br | HTTPSConnectionPool(host='portaldofii.com.br', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='portaldofii.com.br', port=443): Failed to resolve 'portaldofii.com.br' ([Errno 8] nodename nor servna


## 🧡 Seção 10 — Reddit: Fonte #21

**Estratégia de 3 níveis:**
1. PRAW (credenciais `.env`) → coleta completa até 1000 posts
2. API pública `/new.json + /hot.json` → ~200 posts sem autenticação
3. Parquet frozen → dataset pré-coletado commitado

In [11]:

def collect_reddit_live() -> Tuple[List[Dict], bool]:
    records: List[Dict] = []
    try:
        import praw
        reddit = praw.Reddit(
            client_id=REDDIT_CLIENT_ID,
            client_secret=REDDIT_CLIENT_SECRET,
            user_agent=REDDIT_USER_AGENT,
        )
        SEARCH_TERMS = ['FII', 'fundo imobiliário', 'dividendo FII',
                        'renda passiva FII', 'p/vp fii', 'vacância fii']
        for subreddit_name in REDDIT_SUBREDDITS:
            for term in SEARCH_TERMS:
                try:
                    for post in reddit.subreddit(subreddit_name).search(term, limit=100, sort='relevance'):
                        body = post.selftext.strip() if post.selftext.strip() else post.title
                        if not is_fii_relevant(post.title, body):
                            continue
                        url = f'https://reddit.com{post.permalink}'
                        records.append(enforce_schema({
                            'article_id':       make_article_id(url),
                            'source':           'reddit.com',
                            'source_type':      'reddit',
                            'title':            post.title,
                            'content':          body or None,
                            'summary':          body[:400] if body else None,
                            'url':              url,
                            'author':           str(post.author) if post.author else None,
                            'published_at':     datetime.fromtimestamp(
                                                    post.created_utc, tz=timezone.utc
                                                ).isoformat(),
                            'collected_at':     datetime.now(timezone.utc).isoformat(),
                            'language':         'pt-br',
                            'tags':             f'r/{subreddit_name}',
                            'query_used':       term,
                            'ingestion_method': 'praw',
                            'raw_html':         None,
                            'content_hash':     make_content_hash(post.title, body),
                            'metadata_json':    json.dumps({
                                'subreddit': subreddit_name,
                                'score': post.score,
                                'num_comments': post.num_comments,
                                'upvote_ratio': post.upvote_ratio,
                            }),
                        }))
                    time.sleep(RATE_LIMIT_DELAY)
                except Exception as exc:
                    log_source_failure(logger, f'reddit/r/{subreddit_name}', str(exc))
        return records, True
    except ImportError:
        return [], False
    except Exception as exc:
        log_source_failure(logger, 'reddit.com', str(exc))
        return [], False


def collect_reddit_public_api() -> List[Dict]:
    """API pública Reddit: /new.json + /hot.json — sem autenticação."""
    records: List[Dict] = []
    _headers = {
        'User-Agent': 'FIIIntelligencePlatform/1.0 (academic; PUC-SP FACEI)',
        'Accept': 'application/json',
    }
    _fii_kw = {'fii','fiis','fundo','fundos','dividendo','dividendos',
               'provento','proventos','imobiliario','imobiliarios',
               'vacancia','rendimento','cotista','cota','yield','p/vp'}
    _seen: set = set()

    for subreddit in REDDIT_SUBREDDITS:
        for endpoint in ['new', 'hot']:
            try:
                _url    = f'https://www.reddit.com/r/{subreddit}/{endpoint}.json'
                _resp   = requests.get(_url, headers=_headers,
                                       params={'limit': 100}, timeout=REQUEST_TIMEOUT)
                if _resp.status_code == 429:
                    print(f'      ⏳ Rate limit r/{subreddit} — aguardando 15s...')
                    time.sleep(15.0)
                    continue
                if _resp.status_code != 200:
                    print(f'      ⚠️  r/{subreddit}/{endpoint}: HTTP {_resp.status_code}')
                    time.sleep(2.0)
                    continue

                _posts = _resp.json().get('data', {}).get('children', [])
                _added = 0
                for _post in _posts:
                    _p     = _post.get('data', {})
                    _title = (_p.get('title') or '').lower()
                    _body  = (_p.get('selftext') or '').lower()
                    if not any(kw in _title or kw in _body for kw in _fii_kw):
                        continue
                    _url_post = f"https://www.reddit.com{_p.get('permalink','')}"
                    _aid      = hashlib.sha256(_url_post.encode()).hexdigest()
                    if _aid in _seen:
                        continue
                    _seen.add(_aid)
                    _raw_title = _p.get('title', '')
                    _raw_body  = _p.get('selftext', '')
                    _content   = f'{_raw_title}\n{_raw_body}'.strip()
                    _pub = ''
                    if _p.get('created_utc'):
                        _pub = datetime.fromtimestamp(
                            float(_p['created_utc']), tz=timezone.utc
                        ).isoformat()
                    records.append({
                        'article_id':       _aid,
                        'source':           'reddit.com',
                        'source_type':      'reddit',
                        'title':            _raw_title,
                        'content':          _content,
                        'summary':          _raw_body[:500],
                        'url':              _url_post,
                        'author':           str(_p.get('author', '[deleted]')),
                        'published_at':     _pub or None,
                        'collected_at':     datetime.now(timezone.utc).isoformat(),
                        'language':         'pt-br',
                        'tags':             f'r/{subreddit}',
                        'query_used':       f'r/{subreddit}/{endpoint}',
                        'ingestion_method': 'reddit_public_api',
                        'raw_html':         '',
                        'content_hash':     make_content_hash(_raw_title, _content),
                        'metadata_json':    json.dumps({
                            'score': _p.get('score', 0),
                            'num_comments': _p.get('num_comments', 0),
                            'upvote_ratio': _p.get('upvote_ratio', 0.0),
                            'subreddit': subreddit, 'endpoint': endpoint,
                        }),
                    })
                    _added += 1
                print(f'      ✅ r/{subreddit}/{endpoint}: {_added} posts FII')
                time.sleep(2.0)
            except Exception as _e:
                print(f'      ⚠️  r/{subreddit}/{endpoint}: {type(_e).__name__}: {_e}')
                continue
    return records

print('✅ Reddit collectors definidos (PRAW + public API v3)')

✅ Reddit collectors definidos (PRAW + public API v3)


## 🔁 Seção 11 — Execução Reddit / Google News (3 Níveis de Fallback)

> ⚠️ **AVISO — Reddit API HTTP 403 (mudança de política, abril/2023)**
>
> O Reddit desativou o acesso JSON público sem autenticação OAuth.
> Requests para `/new.json` e `/hot.json` retornam **403 Forbidden** sem token.
> Problema amplamente documentado pela comunidade de desenvolvedores.
> Referência: [Reddit API Changes 2023](https://www.reddit.com/r/reddit/comments/145bram/)
>
> **Estratégia de 3 níveis aplicada:**
>
> | Nível | Método | Requisito | Status |
> |---|---|---|---|
> | 1 | PRAW (OAuth) | `REDDIT_CLIENT_ID` + `REDDIT_CLIENT_SECRET` no `.env` | Funciona se credenciais configuradas |
> | 2 | **Google News RSS PT-BR** | Nenhum — feedparser nativo | ✅ Fallback gratuito e estável |
> | 3 | Parquet frozen | `data/external/reddit_fii_posts.parquet` | Dataset pré-coletado |


In [12]:
# ─── Google News RSS — Fallback da Fonte #21 ───────────────────────────────
# Reddit /new.json retorna HTTP 403 desde abril/2023 (API policy change).
# Google News RSS é gratuito, sem autenticação e retorna notícias FII em PT-BR.

GOOGLE_NEWS_RSS_QUERIES: List[str] = [
    "https://news.google.com/rss/search?q=fundo+imobili%C3%A1rio+FII&hl=pt-BR&gl=BR&ceid=BR:pt-419",
    "https://news.google.com/rss/search?q=FII+dividendo+provento&hl=pt-BR&gl=BR&ceid=BR:pt-419",
    "https://news.google.com/rss/search?q=FII+vacancia+inadimplencia&hl=pt-BR&gl=BR&ceid=BR:pt-419",
    "https://news.google.com/rss/search?q=fundos+imobiliarios+renda+passiva&hl=pt-BR&gl=BR&ceid=BR:pt-419",
    "https://news.google.com/rss/search?q=HGLG11+KNRI11+XPML11+FII&hl=pt-BR&gl=BR&ceid=BR:pt-419",
]


def collect_google_news_rss() -> List[Dict]:
    """
    Fallback da Fonte #21: Google News RSS em PT-BR.
    Gratuito, sem autenticação, feedparser nativo.
    Substitui a API pública Reddit desativada (HTTP 403 desde abril/2023).
    """
    records: List[Dict] = []
    seen_hashes: set = set()

    for feed_url in GOOGLE_NEWS_RSS_QUERIES:
        try:
            feed = feedparser.parse(feed_url)
            if feed.bozo and not feed.entries:
                log_source_failure(logger, 'google_news_rss', str(feed.bozo_exception))
                continue

            for entry in feed.entries:
                url_raw = entry.get('link', '').strip()
                title   = entry.get('title', '').strip()
                if not url_raw or not title:
                    continue

                url = normalize_url(url_raw)
                if not is_fii_relevant(title, ''):
                    continue

                c_hash = make_content_hash(title, title)
                if c_hash in seen_hashes:
                    log_duplicate(logger, title)
                    continue
                seen_hashes.add(c_hash)

                summary = strip_html(entry.get('summary', entry.get('description', '')))
                pub_at  = entry.get('published') or entry.get('updated') or None

                # Google News: título formato 'Headline - Portal'
                portal = 'google_news_aggregated'
                if ' - ' in title:
                    portal = title.rsplit(' - ', 1)[-1].strip()[:60]
                    title  = title.rsplit(' - ', 1)[0].strip()

                records.append(enforce_schema({
                    'article_id':       make_article_id(url),
                    'source':           'news.google.com',
                    'source_type':      'reddit',
                    'title':            title,
                    'content':          summary or None,
                    'summary':          summary[:400] if summary else None,
                    'url':              url,
                    'author':           portal,
                    'published_at':     pub_at,
                    'collected_at':     datetime.now(timezone.utc).isoformat(),
                    'language':         'pt-br',
                    'tags':             'google_news_rss',
                    'query_used':       matched_filter_term(title, summary or ''),
                    'ingestion_method': 'feedparser_google_news',
                    'raw_html':         None,
                    'content_hash':     c_hash,
                    'metadata_json':    json.dumps({
                        'original_portal':  portal,
                        'google_news_feed': feed_url[:80],
                    }),
                }))
            time.sleep(RATE_LIMIT_DELAY)
        except Exception as exc:
            log_source_failure(logger, 'google_news_rss', str(exc))
            continue

    return records


print('✅ collect_google_news_rss() definida')
print(f'   {len(GOOGLE_NEWS_RSS_QUERIES)} queries Google News PT-BR configuradas')
print()
print('📌 Reddit HTTP 403: API pública desativada em abril/2023.')
print('   Nível 2 agora usa Google News RSS como substituto estável.')
print('   Ref: https://www.reddit.com/r/reddit/comments/145bram/')

✅ collect_google_news_rss() definida
   5 queries Google News PT-BR configuradas

📌 Reddit HTTP 403: API pública desativada em abril/2023.
   Nível 2 agora usa Google News RSS como substituto estável.
   Ref: https://www.reddit.com/r/reddit/comments/145bram/


In [13]:
print()
print('🧡 REDDIT / GOOGLE NEWS — Behavioral & Social Intelligence Layer (Fonte #21)')
print(f'Subreddits: r/investimentos · r/farialimabets')
print(f'REDDIT_API_AVAILABLE: {REDDIT_API_AVAILABLE}')
print()

reddit_records: List[Dict] = []
reddit_method = 'none'

# ── Nível 1: PRAW (OAuth) ─────────────────────────────────────────────────
if REDDIT_API_AVAILABLE:
    print('   Tentando Nível 1: PRAW (OAuth)...')
    _praw_records, _praw_ok = collect_reddit_live()
    if _praw_ok and _praw_records:
        reddit_records = _praw_records
        reddit_method  = 'praw'
        log_source_success(logger, 'reddit.com/praw', len(reddit_records))
        print(f'   ✅ PRAW: {len(reddit_records)} posts coletados')
    else:
        print('   ⚠️  PRAW retornou 0 registros → tentando Nível 2')
else:
    print('   ⏭️  Nível 1 ignorado — REDDIT_CLIENT_ID não configurado no .env')

# ── Nível 2: Google News RSS (substitui API pública Reddit — HTTP 403) ────
if not reddit_records:
    print()
    print('   Tentando Nível 2: Google News RSS PT-BR...')
    print('   ℹ️  Reddit /new.json retorna HTTP 403 desde abril/2023.')
    print('   ℹ️  Google News RSS: gratuito, sem auth, feedparser nativo.')
    reddit_records = collect_google_news_rss()
    if reddit_records:
        reddit_method = 'google_news_rss'
        log_source_success(logger, 'google_news_rss', len(reddit_records))
        print(f'   ✅ Google News RSS: {len(reddit_records)} artigos FII coletados')
    else:
        print('   ⚠️  Google News RSS: 0 resultados → tentando Nível 3')

# ── Nível 3: Parquet frozen ───────────────────────────────────────────────
if not reddit_records:
    print()
    print('   Tentando Nível 3: parquet frozen...')
    _frozen = Path(EXTERNAL_DIR) / 'reddit_fii_posts.parquet'
    if _frozen.exists():
        reddit_records = pd.read_parquet(_frozen).to_dict(orient='records')
        reddit_method  = 'frozen_parquet'
        log_source_success(logger, 'reddit/frozen', len(reddit_records))
        print(f'   ✅ Frozen: {len(reddit_records)} registros de {_frozen.name}')
    else:
        print()
        print('   ╔════════════════════════════════════════════════════════════╗')
        print('   ║  ⚠️  FONTE #21 — TODOS OS NÍVEIS INDISPONÍVEIS            ║')
        print('   ║                                                            ║')
        print('   ║  Nível 1 PRAW:         REDDIT_CLIENT_ID não configurado  ║')
        print('   ║  Nível 2 Google News:  0 resultados FII nesta execução  ║')
        print('   ║  Nível 3 Frozen:       arquivo não encontrado            ║')
        print('   ║                                                            ║')
        print('   ║  Para ativar PRAW: configure no .env                     ║')
        print('   ║    REDDIT_CLIENT_ID=...                                  ║')
        print('   ║    REDDIT_CLIENT_SECRET=...                              ║')
        print('   ║    https://www.reddit.com/prefs/apps                     ║')
        print('   ║                                                            ║')
        print('   ║  ✅ Pipeline continua normalmente com 20 fontes          ║')
        print('   ╚════════════════════════════════════════════════════════════╝')
        log_source_failure(logger, 'fonte_21',
                           'Reddit 403 + Google News 0 + no frozen parquet')

# ── Construir DataFrame Bronze ────────────────────────────────────────────
df_reddit = (pd.DataFrame(reddit_records, columns=BRONZE_COLUMNS)
             if reddit_records else pd.DataFrame(columns=BRONZE_COLUMNS))
if not df_reddit.empty:
    df_reddit = df_reddit.drop_duplicates('article_id').drop_duplicates('content_hash')

print()
print(f'   📊 Fonte #21 total: {len(df_reddit):,} registros (método: {reddit_method})')
if reddit_method == 'google_news_rss':
    print('   ℹ️  source_type=reddit mantido no schema para compatibilidade downstream.')
    print('   ℹ️  author = portal de origem extraído do título Google News.')


🧡 REDDIT / GOOGLE NEWS — Behavioral & Social Intelligence Layer (Fonte #21)
Subreddits: r/investimentos · r/farialimabets
REDDIT_API_AVAILABLE: False

   ⏭️  Nível 1 ignorado — REDDIT_CLIENT_ID não configurado no .env

   Tentando Nível 2: Google News RSS PT-BR...
   ℹ️  Reddit /new.json retorna HTTP 403 desde abril/2023.
   ℹ️  Google News RSS: gratuito, sem auth, feedparser nativo.
2026-06-14T15:38:37 | INFO     | fii_pipeline.ingestion | SOURCE_OK   | google_news_rss | 345 records
   ✅ Google News RSS: 345 artigos FII coletados

   📊 Fonte #21 total: 345 registros (método: google_news_rss)
   ℹ️  source_type=reddit mantido no schema para compatibilidade downstream.
   ℹ️  author = portal de origem extraído do título Google News.


## 💾 Seção 12 — Consolidação e Freeze do Dataset Bronze

In [14]:

print('\n💾 CONSOLIDAÇÃO E FREEZE DO DATASET BRONZE')
print('=' * 60)

# ── Concatenar todas as fontes ────────────────────────────────────────────────
dfs = [df for df in [df_rss, df_portals, df_reddit] if not df.empty]
if not dfs:
    raise RuntimeError("Nenhuma fonte retornou dados. Verifique conectividade e tente novamente.")

df_bronze_all = pd.concat(dfs, ignore_index=True)
n_before_dedup = len(df_bronze_all)

# ── Dedup final cross-source ──────────────────────────────────────────────────
df_bronze_all = (
    df_bronze_all
    .drop_duplicates('article_id')
    .drop_duplicates('content_hash')
    .reset_index(drop=True)
)
n_after_dedup = len(df_bronze_all)

print(f'\n   Registros antes do dedup final : {n_before_dedup:,}')
print(f'   Registros após dedup final     : {n_after_dedup:,}')
print(f'   Duplicatas removidas           : {n_before_dedup - n_after_dedup:,}')

# ── Salvar parquet por tipo + combinado ───────────────────────────────────────
EXT = Path(EXTERNAL_DIR)
EXT.mkdir(parents=True, exist_ok=True)

paths_written = {}

for source_type, df_part in [
    ('rss',      df_rss),
    ('portal',   df_portals),
    ('reddit',   df_reddit),
]:
    if not df_part.empty:
        p = EXT / f'{source_type}_fii_articles.parquet'
        df_part.to_parquet(p, index=False, compression='snappy')
        paths_written[source_type] = str(p)
        print(f'   💾 {p.name}: {len(df_part):,} registros')

# ── Arquivo combinado (input primário para NB02) ──────────────────────────────
combined_path = EXT / 'bronze_all_articles.parquet'
df_bronze_all.to_parquet(combined_path, index=False, compression='snappy')
paths_written['combined'] = str(combined_path)
print(f'   💾 {combined_path.name}: {n_after_dedup:,} registros (COMBINED)')

log_dataset_frozen(logger, str(combined_path), n_after_dedup)

# ── Relatório de proveniência ─────────────────────────────────────────────────
print('\n📊 Relatório de proveniência por fonte:')
print(f'   {"Fonte":<40} {"Tipo":<12} {"Registros":>10}')
print('   ' + '-' * 65)
for src_type in ['rss', 'scraping', 'reddit']:
    subset = df_bronze_all[df_bronze_all['source_type'] == src_type]
    by_source = subset.groupby('source').size().sort_values(ascending=False)
    for src, cnt in by_source.items():
        print(f'   {src:<40} {src_type:<12} {cnt:>10,}')

print('\n✅ Dataset Bronze congelado. NB02–NB07 lêem apenas data/external/')
print(f'   Total: {n_after_dedup:,} artigos únicos de 21 fontes')
print('   🔒 FREEZE — sem novos requests HTTP nos notebooks downstream')


💾 CONSOLIDAÇÃO E FREEZE DO DATASET BRONZE

   Registros antes do dedup final : 503
   Registros após dedup final     : 503
   Duplicatas removidas           : 0
   💾 rss_fii_articles.parquet: 69 registros
   💾 portal_fii_articles.parquet: 89 registros
   💾 reddit_fii_articles.parquet: 345 registros
   💾 bronze_all_articles.parquet: 503 registros (COMBINED)
2026-06-14T15:38:38 | INFO     | fii_pipeline.ingestion | DATASET_FROZEN | /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/external/bronze_all_articles.parquet | 503 records | downstream reads only

📊 Relatório de proveniência por fonte:
   Fonte                                    Tipo          Registros
   -----------------------------------------------------------------
   empiricus.com.br                         rss                  32
   cnnbrasil.com.br                         rss                  17
   seudinheiro.com                          rss                   6
   sunoresearch.com.br         


## ✅ NB01 Completo

| Artefato | Localização |
|----------|-------------|
| `bronze_all_articles.parquet` | `data/external/` |
| `rss_fii_articles.parquet` | `data/external/` |
| `portal_fii_articles.parquet` | `data/external/` |
| `reddit_fii_posts.parquet` | `data/external/` |

▶️ **Próximo:** `02_bronze_to_silver.ipynb`